In [4]:
# Initial preprocessing pipeline with numeric RUL labels only, then export to CSV
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np

# -----------------------------
# 1) Load raw competition data
# -----------------------------
train_raw = pd.read_csv('../data/nasa-data/train_FD002.txt', sep=r'\s+', header=None)
test_raw = pd.read_csv('../data/nasa-data/test_FD002.txt', sep=r'\s+', header=None)

# Keep only valid columns (drop trailing whitespace columns if present)
train_raw = train_raw.iloc[:, :26].copy()
test_raw = test_raw.iloc[:, :26].copy()

base_cols = [
    'id', 'cycle', 'setting1', 'setting2', 'setting3',
    's1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10',
    's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21'
 ]
train_raw.columns = base_cols
test_raw.columns = base_cols

# ----------------------------------
# 2) Create numeric RUL labels only
# ----------------------------------
# Train RUL: max cycle per engine - current cycle
train_max_cycle = train_raw.groupby('id', as_index=False)['cycle'].max().rename(columns={'cycle': 'max_cycle'})
train_proc = train_raw.merge(train_max_cycle, on='id', how='left')
train_proc['RUL'] = train_proc['max_cycle'] - train_proc['cycle']
train_proc = train_proc.drop(columns=['max_cycle'])

# Test RUL: use NASA ground truth file for remaining life after last observed cycle
truth_df = pd.read_csv('../data/nasa-data/RUL_FD001.txt', sep=r'\s+', header=None).iloc[:, :1].copy()
truth_df.columns = ['more']
truth_df['id'] = np.arange(1, len(truth_df) + 1)

test_last_cycle = test_raw.groupby('id', as_index=False)['cycle'].max().rename(columns={'cycle': 'last_cycle'})
truth_df = truth_df.merge(test_last_cycle, on='id', how='left')
truth_df['max_cycle'] = truth_df['last_cycle'] + truth_df['more']
truth_df = truth_df[['id', 'max_cycle']]

test_proc = test_raw.merge(truth_df, on='id', how='left')
test_proc['RUL'] = test_proc['max_cycle'] - test_proc['cycle']
test_proc = test_proc.drop(columns=['max_cycle'])

# ------------------------------------------
# 3) Normalize feature columns (keep RUL raw)
# ------------------------------------------
train_proc['cycle_norm'] = train_proc['cycle']
test_proc['cycle_norm'] = test_proc['cycle']

cols_to_normalize = [
    c for c in train_proc.columns
    if c not in ['id', 'cycle', 'RUL']
 ]

scaler = MinMaxScaler()
train_norm = pd.DataFrame(
    scaler.fit_transform(train_proc[cols_to_normalize]),
    columns=cols_to_normalize,
    index=train_proc.index
)
test_norm = pd.DataFrame(
    scaler.transform(test_proc[cols_to_normalize]),
    columns=cols_to_normalize,
    index=test_proc.index
)

train_processed_rul = train_proc[['id', 'cycle', 'RUL']].join(train_norm)
test_processed_rul = test_proc[['id', 'cycle', 'RUL']].join(test_norm)

# Reorder columns so RUL is the last column
ordered_cols = [c for c in train_processed_rul.columns if c != 'RUL'] + ['RUL']
train_processed_rul = train_processed_rul[ordered_cols]
test_processed_rul = test_processed_rul[ordered_cols]

# ----------------
# 4) Export to CSV
# ----------------
out_dir = Path('../data/processed-nasa-data')
out_dir.mkdir(parents=True, exist_ok=True)

train_out_path = out_dir / 'train_processed_rul_only_fd002.csv'
test_out_path = out_dir / 'test_processed_rul_only_fd002.csv'

train_processed_rul.to_csv(train_out_path, index=False)
test_processed_rul.to_csv(test_out_path, index=False)

print(f'Saved: {train_out_path}  shape={train_processed_rul.shape}')
print(f'Saved: {test_out_path}  shape={test_processed_rul.shape}')
print('Label format: numeric RUL only (no binary label columns).')

Saved: ..\data\processed-nasa-data\train_processed_rul_only_fd002.csv  shape=(53759, 28)
Saved: ..\data\processed-nasa-data\test_processed_rul_only_fd002.csv  shape=(33991, 28)
Label format: numeric RUL only (no binary label columns).
